# Solvers SAT (PySAT)

## Setup

In [1]:
import time, lzma, gzip, bz2
from pysat.formula import CNF
from pysat.solvers import Solver


def _open_text(path):
    if path.endswith((".xz", ".lzma")): return lzma.open(path, "rt")
    if path.endswith(".gz"): return gzip.open(path, "rt")
    if path.endswith(".bz2"): return bz2.open(path, "rt")
    return open(path, "r")


def load_cnf(path):
    cnf, clause = CNF(), []
    with _open_text(path) as f:
        for line in f:
            line = line.strip()
            if not line or line[0] in "cp%":
                continue
            for tok in line.split():
                lit = int(tok)
                if lit == 0:
                    if clause:
                        cnf.append(clause); clause = []
                else:
                    clause.append(lit)
    if clause:
        cnf.append(clause)
    return cnf

## Resolver una instancia SAT

In [ ]:
cnf = load_cnf("../data/satlib/random_3sat/uf100-01.cnf")
print(f"{cnf.nv} variables, {len(cnf.clauses)} clausulas")

with Solver(name="glucose4", bootstrap_with=cnf) as s:
    sat = s.solve()
    model = s.get_model() if sat else None

print("resultado:", "SAT" if sat else "UNSAT")
print("primeros 10 literales del modelo:", model[:10])

100 variables, 430 cláusulas
resultado: SAT
primeros 10 literales del modelo: [-1, -2, -3, 4, 5, -6, 7, 8, 9, 10]


In [4]:
model

[-1,
 -2,
 -3,
 4,
 5,
 -6,
 7,
 8,
 9,
 10,
 11,
 -12,
 13,
 14,
 15,
 -16,
 -17,
 -18,
 -19,
 20,
 21,
 -22,
 -23,
 -24,
 25,
 -26,
 -27,
 -28,
 29,
 -30,
 -31,
 32,
 33,
 -34,
 -35,
 36,
 -37,
 38,
 39,
 40,
 -41,
 -42,
 43,
 44,
 45,
 46,
 -47,
 -48,
 49,
 50,
 51,
 52,
 53,
 -54,
 55,
 56,
 -57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 -70,
 71,
 72,
 -73,
 74,
 75,
 -76,
 -77,
 -78,
 79,
 80,
 -81,
 -82,
 83,
 84,
 85,
 -86,
 87,
 -88,
 -89,
 -90,
 -91,
 -92,
 -93,
 -94,
 95,
 96,
 97,
 98,
 99,
 -100]

Verificamos a mano que el modelo realmente satisface el CNF:

In [6]:
assign = {abs(l): (l > 0) for l in model}
ok = all(any((lit > 0) == assign[abs(lit)] for lit in clause) for clause in cnf.clauses)
ok

True

## Instancia UNSAT

In [7]:
cnf_unsat = load_cnf("../data/satlib/random_3sat/UUF100.430.1000/uuf100-0100.cnf")
with Solver(name="glucose4", bootstrap_with=cnf_unsat) as s:
    print("resultado:", "SAT" if s.solve() else "UNSAT")

resultado: UNSAT


## Comparando Solvers


In [9]:
engines = ["glucose3", "glucose4", "cadical153", "cadical195", "minisat22", "kissat404"]
cnf = load_cnf("../data/satlib/random_3sat/uf100-01.cnf")

print(f"{'solver':12} {'resultado':10} {'tiempo':>10}")
for name in engines:
    t = time.time()
    with Solver(name=name, bootstrap_with=cnf) as s:
        sat = s.solve()
    print(f"{name:12} {'SAT' if sat else 'UNSAT':10} {(time.time()-t)*1000:8.1f} ms")

solver       resultado      tiempo
glucose3     SAT             0.8 ms
glucose4     SAT             0.7 ms
cadical153   SAT             0.5 ms
cadical195   SAT             0.5 ms
minisat22    SAT             0.5 ms
kissat404    SAT            15.5 ms


## Cargando una instancia comprimida del holdout (.xz)

In [16]:
import os

xz = "../data/satcomp_holdout/2024/sgen1-unsat-121-100.cnf.xz"
if os.path.exists(xz):
    c = load_cnf(xz)
    print(f"{os.path.basename(xz)}: {c.nv} variables, {len(c.clauses)} cláusulas")

sgen1-unsat-121-100.cnf.xz: 121 variables, 252 cláusulas


In [15]:
if os.path.exists(xz):
    c = load_cnf(xz)
    with Solver(name="cadical195", bootstrap_with=c, use_timer=True) as s:
        s.conf_budget(10000)         
        res = s.solve_limited()        
        verdict = {True: "SAT", False: "UNSAT", None: "INDET (agotó presupuesto)"}[res]
        print(f"{os.path.basename(xz)} -> {verdict}  ({s.time():.3f}s)")

sgen1-unsat-121-100.cnf.xz -> INDET (agotó presupuesto)  (0.105s)
